# ETP Semantic Drift Experiment — Core Setup

This notebook is structured around the **5-component experimental setting**:

1. **Intended theory** — an equation from ETP
2. **Representation change** — one LLM call transforms it into another form
3. **Generated output** — classify what the model produced (valid? which drift type?)
4. **Semantic validation** — compare against the intended equation using 2+ methods
5. **Analysis** — summarize where drift happened and what each method caught

We build this **incrementally**: Section A gets the single LLM call working and verified in isolation before anything else depends on it. Only after that works do we chain it into the full pipeline.


## Setup: Drive, dependencies, API key

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORKDIR = '/content/etp_pipeline'
DRIVE_CACHE = '/content/drive/MyDrive/etp_pipeline'  # caches the built implication matrix between runtimes
os.makedirs(WORKDIR, exist_ok=True)
os.makedirs(DRIVE_CACHE, exist_ok=True)
%cd $WORKDIR


In [ ]:
!pip install -q transformers accelerate


## Get the oracle (the shared `oracle/` package -- no embedded copy)

The normalizer / node mapper / implication-graph oracle used below is the SAME code the rest of the project uses: the [`oracle/` package](https://github.com/corpaci/semantic-diffchecking/tree/main/oracle). This notebook previously carried its own embedded implementation (`etp_oracle.py` + `find_equation_id.py`); that copy is gone, so there is exactly one oracle to maintain.

The only piece still defined in-notebook is the brute-force finite-magma fallback (two cells down), which the package does not have yet.


In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/corpaci/semantic-diffchecking.git'
REPO_DIR = os.path.join(WORKDIR, 'semantic-diffchecking')
ORACLE_DIR = os.path.join(REPO_DIR, 'oracle')

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

if ORACLE_DIR not in sys.path:
    sys.path.insert(0, ORACLE_DIR)

# The package locates the two ETP data files through this variable.
# Set it BEFORE importing any oracle module -- mapper.py reads it at import time.
os.environ['ETP_ROOT'] = os.path.join(WORKDIR, 'equational_theories')

import normalizer, mapper, build_matrix  # noqa: F401  (fail fast if the path is wrong)
print('oracle package importable from', ORACLE_DIR)


In [ ]:
# Brute-force finite-magma fallback -- the ONE piece not in oracle/ yet.
# Used when the generated equation parses but is outside the 4,694-law
# fragment (order > 4), so there is no graph node to look up.
#
# It searches every multiplication table of size 2..max_size for a magma
# where one law holds and the other fails. A witness REFUTES an implication;
# absence of a witness proves nothing (evidence, not certainty). Hence the
# only labels it can ever justify are 'incomparable' (both directions
# refuted) and 'unknown' (anything else).
import itertools

from normalizer import Var, parse_equation


def _all_magmas(n):
    """Every n x n Cayley table on {0..n-1} -- n^(n^2) of them, so tiny n only."""
    for entries in itertools.product(range(n), repeat=n * n):
        yield [entries[i * n:(i + 1) * n] for i in range(n)]


def _eval_term(term, env, table):
    if isinstance(term, Var):
        return env[term.name]
    return table[_eval_term(term.left, env, table)][_eval_term(term.right, env, table)]


def _satisfies(eq, table, n):
    """Does this magma satisfy the law for every variable assignment?"""
    varnames = []
    eq.lhs.variables(varnames)
    eq.rhs.variables(varnames)
    for assignment in itertools.product(range(n), repeat=len(varnames)):
        env = dict(zip(varnames, assignment))
        if _eval_term(eq.lhs, env, table) != _eval_term(eq.rhs, env, table):
            return False
    return True


def brute_force_relation(eq_a, eq_b, max_size=3):
    """Search magmas of size 2..max_size for counterexamples to A=>B and B=>A.

    eq_a / eq_b: equation strings or already-parsed normalizer.Equation ASTs.
    Returns a_implies_b / b_implies_a as False (refuted, witness attached) or
    None (no witness up to max_size -> inconclusive). Never True: this search
    can refute an implication, never prove one.
    """
    if isinstance(eq_a, str):
        eq_a = parse_equation(eq_a)
    if isinstance(eq_b, str):
        eq_b = parse_equation(eq_b)

    result = {'a_implies_b': None, 'b_implies_a': None,
              'witness_a_not_b': None, 'witness_b_not_a': None}
    for n in range(2, max_size + 1):
        for table in _all_magmas(n):
            a_holds = _satisfies(eq_a, table, n)
            b_holds = _satisfies(eq_b, table, n)
            if a_holds and not b_holds and result['a_implies_b'] is None:
                result['a_implies_b'] = False
                result['witness_a_not_b'] = table
            if b_holds and not a_holds and result['b_implies_a'] is None:
                result['b_implies_a'] = False
                result['witness_b_not_a'] = table
        if result['a_implies_b'] is False and result['b_implies_a'] is False:
            break
    return result

print('brute_force_relation defined (fallback oracle, refute-only)')


## Get the ETP data + build the implication matrix

This is your **reference / ground truth** -- the pre-verified implication graph from the Equational Theories Project, compiled by `oracle/build_matrix.py` into a compact 22 MB binary matrix (much lighter than the old 1 GB SQLite edge-list database). Built once (~1-2 min) and cached to Drive.


In [ ]:
!git clone --depth 1 https://github.com/teorth/equational_theories.git


In [ ]:
import os, shutil, subprocess, sys
from build_matrix import MATRIX_BIN, META_JSON

cached = {p: os.path.join(DRIVE_CACHE, os.path.basename(p)) for p in (MATRIX_BIN, META_JSON)}

if all(os.path.exists(c) for c in cached.values()):
    print('Restoring implication matrix from Drive (no rebuild needed)...')
    os.makedirs(os.path.dirname(MATRIX_BIN), exist_ok=True)
    for local, drive_copy in cached.items():
        shutil.copy(drive_copy, local)
else:
    print('Building implication matrix from scratch (~1-2 min, one-time)...')
    # Run as a script: build_matrix.main() reads sys.argv, which in a notebook
    # belongs to the kernel, so a subprocess keeps the arg handling clean.
    subprocess.run([sys.executable, os.path.join(ORACLE_DIR, 'build_matrix.py')], check=True)
    for local, drive_copy in cached.items():
        shutil.copy(local, drive_copy)

from oracle import SemanticOracle
oracle = SemanticOracle()
print(f'Ready: {oracle.n} laws, matrix {os.path.getsize(MATRIX_BIN) / 1e6:.0f} MB')


---
# Section A — Load an open model LOCALLY (not a hosted API)

Using local weights instead of a hosted API is a deliberate choice here: if you plan to do interpretability work later (probing hidden states, activation patching, steering vectors), you need **direct access to the model's internals**, which a hosted API can never give you, no matter how permissions are configured. This cell loads the model straight into the Colab GPU's memory and exposes `model` and `tokenizer` as top-level variables you can reach into from later cells.

**Model choice**: `Qwen/Qwen2.5-1.5B-Instruct` — small enough to run comfortably on Colab's free-tier T4 GPU (~15GB VRAM), no gated-access approval needed (unlike some Llama checkpoints), and capable enough for short equation/description tasks. Swap `MODEL_NAME` for something else if you outgrow it.

**If you're planning to use TransformerLens** for the probing/steering work specifically, check TransformerLens's current supported-model list before committing to Qwen -- support varies by architecture and version, and a GPT-2/Llama/Mistral/Gemma-family model may be a smoother fit depending on what version of TransformerLens you're on.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'  # change here if you need a different model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()
print('Loaded', MODEL_NAME, 'on', model.device)
print('model and tokenizer are now available as top-level variables --')
print('use these directly for hooks / hidden-state access / steering later.')


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-1.5B-Instruct on cuda:0
model and tokenizer are now available as top-level variables --
use these directly for hooks / hidden-state access / steering later.


### The single call_llm() function -- everything downstream goes through this

Runs generation locally (no network call, no API key, no permissions to configure). Deterministic (`do_sample=False`) so results are reproducible run to run -- useful when you're later comparing against probed/steered variants of the same model.

In [ ]:
def call_llm(prompt: str, max_new_tokens: int = 200) -> str:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,          # <- explicitly request dict form
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,               # <- unpack input_ids AND attention_mask
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[-1]:]   # <- index into the dict
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

### Sanity check: a trivial, throwaway call
Confirms the model loaded and generates sensible text before we trust it with real data.

In [ ]:
test_reply = call_llm("Reply with exactly the word: OK")
print(repr(test_reply))
print('Section A passed: local model is generating text.')


'OK'
Section A passed: local model is generating text.


---
# Section B — Component 1: Intended theory

Pick a real equation from ETP by its catalogue number. This is your ground-truth starting point -- everything downstream is measured against this exact equation.

In [ ]:
INTENDED_ID = 4512  # associativity -- change this to try a different equation
intended_equation = oracle.mapper.law_text(INTENDED_ID)
print(f'Intended theory: E{INTENDED_ID} = {intended_equation}')


---
# Section C — Component 2: Representation change (the single LLM call, applied)

We do the **equation -> natural language -> equation** round trip, using the SAME `call_llm()` function from Section A for both hops. This is the representation-change step the framework describes -- specifically the 'natural language to equation' direction is the one we're studying for drift, so we generate the NL description first, then translate it back blind (the translation call never sees the original equation, only the description).

In [ ]:
def describe_equation_nl(equation_str: str) -> str:
    """Representation change, direction 1: equation -> natural language."""
    prompt = (
        "Describe the following magma equation in plain, unambiguous natural "
        "language, as you would to a mathematician who cannot see the formula. "
        "Do not use the '◇' symbol or any formal notation in your description.\n\n"
        f"Equation: {equation_str}"
    )
    return call_llm(prompt, max_new_tokens=400)

def translate_nl_to_equation(nl_description: str) -> str:
    """Representation change, direction 2: natural language -> equation.
    This call ONLY sees the description -- never the original equation --
    which is what makes any mismatch meaningful.

    NOTE: smaller/open models are much less reliable than a frontier model
    at following 'output ONLY the equation' instructions -- they commonly
    wrap output in LaTeX (\\( \\), \\[ \\]) and use LaTeX command names
    (\\oplus) instead of the literal '◇' character. The prompt below is
    written to reduce that with an explicit negative example and a
    one-shot demonstration; normalize_equation() ALSO cleans this up
    automatically as a safety net, so a stray LaTeX wrapper won't
    silently fail the whole row.
    """
    prompt = (
        "Translate the following natural-language description of a law for a "
        "binary operation into a single formal equation.\n\n"
        "STRICT OUTPUT RULES:\n"
        "- Use the literal character ◇ for the operation -- never \\oplus, \\cdot, *, or any LaTeX command.\n"
        "- Do NOT wrap the equation in \\( \\), \\[ \\], $ $, or any other delimiters.\n"
        "- Do NOT add any preamble like 'The equation is:' or 'Sure, here it is:'.\n"
        "- Output ONLY the bare equation and nothing else.\n\n"
        "Example -- if the description were 'combining two elements gives the same "
        "result regardless of order', your ENTIRE response should be exactly:\n"
        "x ◇ y = y ◇ x\n\n"
        f"Description: {nl_description}\n\n"
        "Your answer (bare equation only):"
    )
    return call_llm(prompt)


In [ ]:
nl_description = describe_equation_nl(intended_equation)
print('NL description (from the LLM):')
print(' ', nl_description)

generated_equation_raw = translate_nl_to_equation(nl_description)
print('\nGenerated equation (translated back by the LLM, blind to the original):')
print(' ', generated_equation_raw)


NL description (from the LLM):
  The given equation is an example of a binary operation called "◇" applied twice on three variables. Let's break it down:

1. The left side of the equation, \( x \) ◇ \( (y \) ◇ \( z ) \), means that we first apply the operation ◇ to \( y \) and \( z \). This gives us some result, which we'll call \( w \).

2. Then, we take this result \( w \) and apply the operation ◇ again to \( x \). So, the right side of the equation, \( (x \) ◇ \( y) \) ◇ \( z \), means that we first apply the operation ◇ to \( x \) and \( y \), giving us another result, which we'll call \( v \). We then take this new result \( v \) and apply the operation ◇ to \( z \).

3. Therefore, the entire equation states that applying the operation ◇ twice in succession between two different pairs of variables results in the same outcome as applying the operation ◇ once between those two variables and then between the other pair of variables.

In simpler terms, if we have three numbers \( x \

---
# Section D — Component 3: Generated output classification

Before comparing meanings, check the basic shape of what came back:
- Did it even parse as a valid equation?
- If valid, does it match one of ETP's 4,694 catalogued laws ("in fragment"), or is it a valid equation that just isn't catalogued ("outside fragment")?

In [ ]:
generated_equation_raw

'\\( (x \\) ◇ \\( y) \\) ◇ \\( z = x \\) ◇ \\( (y \\) ◇ \\( z) \\)'

In [ ]:
gen_map = oracle.mapper.map(generated_equation_raw)

print('Normalized form:', gen_map.normalized)
print('Status:', gen_map.status)   # mapped / parse-failure / outside-fragment / internal-error
print('Matched ETP id:', gen_map.node)
if not gen_map.mapped:
    print('Reason:', gen_map.reason)


---
# Section E — Component 4: Semantic validation (2+ methods)

**Method 1 — Implication graph lookup** (primary, exact): checks the ETP graph directly. Mutual implication = equivalent; one-way = stronger/weaker; neither = incomparable.

**Method 2 — Brute-force finite magma search** (fallback / second method): used automatically when the generated equation is outside the 4,694-law fragment, so there's no graph edge to look up. Searches small magmas (size 2-3) for a concrete counterexample. Can only ever disprove, never prove -- absence of a counterexample is evidence, not certainty.

Using both satisfies the "2+ methods" requirement: graph lookup when possible, brute-force as an independent check or fallback when it isn't.

In [ ]:
verdict = oracle.compare(INTENDED_ID, generated_equation_raw)

if verdict.label == 'parse-failure':
    label = 'parse-failure'
    method_used = 'none (parse failure)'

elif verdict.label == 'outside-fragment':
    # Method 2: brute force, since there's no graph node to look up. Only
    # possible when the output PARSED (a valid identity of order > 4): a
    # multi-operator or otherwise unreadable output leaves no AST to evaluate.
    if gen_map.equation is not None:
        bf = brute_force_relation(intended_equation, gen_map.equation, max_size=3)
        method_used = 'brute_force (outside fragment)'
        # Brute force can only REFUTE an implication (by exhibiting a witness
        # magma), never prove one -- so a single refuted direction cannot justify
        # 'weaker'/'stronger': those labels require the opposite direction PROVEN.
        # Only both-directions-refuted is conclusive.
        if bf['a_implies_b'] is False and bf['b_implies_a'] is False:
            label = 'incomparable'
        else:
            label = 'unknown'
        print('Brute-force detail:', bf)
    else:
        label = 'outside-fragment'
        method_used = 'none (no AST to brute-force)'

elif verdict.label == 'internal-error':
    label = 'internal-error'
    method_used = 'none (normalizer bug -- report this input)'

else:
    # Method 1: implication graph lookup (via the shared SemanticOracle)
    label = verdict.label
    method_used = 'implication_graph'
    # Cross-check with Method 2 anyway, as a second independent signal:
    bf = brute_force_relation(intended_equation, gen_map.equation, max_size=2)
    print('Cross-check via brute-force (size<=2):', bf)

print()
print('=== RESULT ===')
print('Intended:  ', intended_equation)
print('Generated: ', generated_equation_raw)
print('Normalized:', gen_map.normalized)
print('Method used:', method_used)
print('Label:', label)
print('Evidence:', verdict.evidence or verdict.notes)


---
# Section F — Component 5: Analysis

For a single run, analysis is just reading the result above and asking: *was there drift, and if so, what kind?* Once you loop this over many equations (next section), this becomes a real summary.


In [ ]:
if label == 'equivalent':
    print('No semantic drift detected -- the translation preserved meaning.')
elif label in ('weaker', 'stronger'):
    print(f'Directional drift detected: generated equation is {label} than intended.')
elif label == 'incomparable':
    print('The generated equation is unrelated to the intended one -- a genuine mismatch,')
    print('even though it may be syntactically valid and even a real catalogued ETP law.')
elif label == 'parse-failure':
    print('The LLM failed to produce a parseable equation at all.')
elif label == 'outside-fragment':
    print('The LLM produced something outside the one-binary-operation fragment')
    print('(e.g. two different operator symbols) -- the oracle refuses to speak.')
else:
    print(f'Label: {label} -- inspect manually.')


---
# Section G — Loop this over many equations (once Sections A-F work)

Only run this after confirming the single-equation flow above works exactly as expected. This loops Sections C-F over your full equation list.

In [ ]:
import csv


def build_explanation(intended_id, intended_eq, nl, gen_raw, gen_map, label, oracle_used):
    """Builds a human-readable narrative explaining this row's result."""
    lines = []
    lines.append(f'Intended theory: E{intended_id} = "{intended_eq}"')
    lines.append(f'NL description generated: "{nl}"')
    lines.append(f'LLM\'s translation back to a formal equation: "{gen_raw}"')

    if label == 'parse-failure':
        lines.append("Normalization failed: the LLM's output could not be parsed as a valid equation.")
        lines.append('Semantic drift: UNRESOLVABLE -- no comparison possible, the translation step itself failed.')
        return ' | '.join(lines)

    lines.append(f'Normalized form: "{gen_map.normalized}"' +
                 (f' (matches ETP E{gen_map.node})' if gen_map.mapped else ' (not a catalogued ETP law)'))
    lines.append(f'Comparison method used: {oracle_used}')

    if label == 'equivalent':
        drift = "NONE -- the LLM's translation preserved the intended meaning exactly."
    elif label == 'stronger':
        drift = ('STRENGTHENING -- the generated equation implies the intended one, but not vice versa. '
                 "The LLM's translation added a constraint that was not present in the original.")
    elif label == 'weaker':
        drift = ('WEAKENING -- the intended equation implies the generated one, but not vice versa. '
                 "The LLM's translation dropped a constraint that was present in the original.")
    elif label == 'incomparable':
        drift = ('INCOMPARABLE -- neither equation implies the other. '
                 "The LLM's translation describes a genuinely different, unrelated law, "
                 'even though it may be syntactically valid.')
    elif label == 'unknown':
        drift = 'UNKNOWN -- the available oracle could not determine the relationship within its search bounds.'
    elif label == 'outside-fragment':
        drift = ('OUTSIDE FRAGMENT -- the output is not a single-binary-operation identity, '
                 'so the formal oracle refuses to speak.')
    else:
        drift = f'{label} (unrecognized label -- inspect manually).'

    lines.append(f'Semantic drift: {drift}')
    return ' | '.join(lines)


def run_one(intended_id, oracle):
    intended_eq = oracle.mapper.law_text(intended_id)
    nl = describe_equation_nl(intended_eq)
    gen_raw = translate_nl_to_equation(nl)
    gen_map = oracle.mapper.map(gen_raw)
    verdict = oracle.compare(intended_id, gen_raw)

    row = {'intended_id': intended_id, 'intended_equation': intended_eq,
           'nl_description': nl, 'generated_equation_raw': gen_raw,
           'normalized_equation': gen_map.normalized, 'generated_status': gen_map.status,
           'generated_etp_id': gen_map.node, 'oracle_used': None, 'label': None,
           'evidence': verdict.evidence or verdict.notes, 'explanation': None}

    if verdict.label == 'parse-failure':
        row['label'] = 'parse-failure'; row['oracle_used'] = 'none (parse failure)'
    elif verdict.label == 'outside-fragment' and gen_map.equation is not None:
        bf = brute_force_relation(intended_eq, gen_map.equation, max_size=3)
        row['oracle_used'] = 'brute_force'
        # brute force can only refute, never prove -- see the Section E note
        if bf['a_implies_b'] is False and bf['b_implies_a'] is False:
            row['label'] = 'incomparable'
        else:
            row['label'] = 'unknown'
    elif verdict.label in ('outside-fragment', 'internal-error'):
        row['label'] = verdict.label; row['oracle_used'] = 'none'
    else:
        row['label'] = verdict.label; row['oracle_used'] = 'implication_graph'

    row['explanation'] = build_explanation(intended_id, intended_eq, nl, gen_raw, gen_map,
                                           row['label'], row['oracle_used'])
    return row

# Load your equation list from equation_selection.csv if you've uploaded one,
# otherwise fall back to a small built-in sample.
import os
if os.path.exists('/content/equation_selection.csv'):
    with open('/content/equation_selection.csv') as f:
        equation_ids = [int(r['id']) for r in csv.DictReader(f)]
else:
    equation_ids = [1, 2, 43, 4512, 168]
    print('No equation_selection.csv found -- using a small built-in sample. '
          'Upload your CSV and re-run this cell to use your full list.')

results = []
for eq_id in equation_ids:
    print(f'Processing E{eq_id}...')
    r = run_one(eq_id, oracle)
    results.append(r)
    print(f'  -> {r["label"]}')
    print(f'  {r["explanation"]}\n')

with open('dataset.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)

print(f'\nWrote {len(results)} rows to dataset.csv')


## Download your results

In [ ]:
from google.colab import files
files.download('dataset.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>